# 查询表达与漏检诊断

我用人工给定的检索表达诊断U03漏检，保留原问题，只请求了一次模型。以下保留实际运行输出；重新运行会生成新记录。第9节首次检查仍为4/5，本次诊断单独记录。

## 原问题与检索表达

原问题是电池放在仓库里是否不用再充电；检索表达是“电池贮存期间的充电要求和充电周期”。后者只用于找资料，回答模型仍收到原问题。表达里没有填写答案数字或页码，目前没有自动查询改写。

先安装仓库根目录依赖，打开本Notebook并选择对应Python内核。

In [1]:
import json
try:
    import manual_project as mp
    import retrieval_upgrade as up
    import query_diagnosis as qd
except ModuleNotFoundError:
    from public_manual import manual_project as mp
    from public_manual import retrieval_upgrade as up
    from public_manual import query_diagnosis as qd

records = mp.prepare_documents()
model = mp.load_search_model()
doc_vectors = mp.encode_documents(records, model)
tools = up.build_search_tools(records, model, doc_vectors)

# 读取已核对的第9节首次结果，不重新请求旧回答。
old_path = mp.BASE_DIR / "results" / "llm09_new5_20261013_121225_300020.json"
old_results = json.loads(old_path.read_text(encoding="utf-8"))
old_u03 = next(result for result in old_results if result["id"] == "U03")
search_query = "电池贮存期间的充电要求和充电周期"
trial = qd.prepare_trial(old_u03, search_query, tools)
print("原问题：", trial["question"])
print("检索表达：", trial["search_query"])


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

原问题： 电池模块放在仓库里就不用再充电了，对吗？手册对此怎么规定？
检索表达： 电池贮存期间的充电要求和充电周期


## 对比候选原文

旧候选来自第9节首次记录，新候选来自人工调整后的表达。这格只检索，不请求DeepSeek。

In [2]:
print("旧候选：", [hit["chunk_id"] for hit in trial["old_retrieved"]])
print("新候选：", [hit["chunk_id"] for hit in trial["retrieved"]])
up.show_hits(trial["retrieved"])


旧候选： ['p06-c02', 'p15-c02', 'p06-c01']
新候选： ['p04-c02', 'p15-c02', 'p14-c02']
EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第4页，块02
用 UPS。
注：贮存期间每六个月给电池模
块充电一次。
贮存 -20 到 60°C
海拔高度 工作 0 - 2,000 m: 正常操作
> 2,000 m：高度每增加 100 m 负载减小
1%
> 3,000 m：UPS 不工作
贮存 0 - 15,000 m
湿度 0 到 95%相对湿度，非冷凝
国际防护等级标准 IP20
供电系统类型 TT 和 TN
污染等级 2
过压范围 II
适用标准 IEC 62040-1
物理规格
塔式型号
UPS 型号 SRV1KI SRV2KI SRV3KI
带包装的尺寸
宽

EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第15页，块02
电压处于
所配置输出电压的±5%范围内，和所
配置输出频率的±3 Hz 范围内，所连
设备即会通过旁路继电器接收市电输
入电源。在模式下逆变器关闭。
如果市电输入超出范围，逆变器会打
开。负载转移到在线模式或电池模
式。所连设备的供电可能中断最长 10
毫秒。
重新启动设置的
最小电池容量
0% 0%, 15%,50%,90%, 在电池的充电量能够提供此设置配置的
运行时间之前，UPS 输出不会打开。
如果配置为 0%，则在市电恢复后，
UPS 输出立即打开。
电池电量不足状
态指示设置
2 分钟 2 分钟、5 分

EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第14页，块02
UMBER, C
电池充电状态 % BAT, %
负载水平百分比(最大瓦数或 VA) % OUT, %
负载水平(kVA) kVA OUT, kVA
所连电池的总 Ah 容量 Ah BAT, Ah
剩余电池运行时间 分钟 BAT, 分
配置
配置 UPS 参数
按照以下步骤配置 UPS 中的参数:
1. 按确认按钮。
2. 按向上/向下按钮导航至“设置” 。
3. 按确认按钮。
4. 使用向上/向下按钮浏览参数。
5. 按确认按钮编辑参数。图标开始闪烁指示编辑。
6. 按向上/向下按钮浏览可用于所选参数的选项。


### 找到的依据

第4页块02包含贮存期间每六个月充电一次的条款。

In [ ]:
# 在这里写块编号和你的预测
## 新候选p04-c02包含贮存期间每六个月充电一次的依据，预测模型能纠正原问题的错误前提并回答充电周期。

## 用新资料回答原问题

下一格只请求DeepSeek一次，保留模型、提示词与最终3块的数量。回答仍针对原问题，结果另存为新文件；原检查记录保留。

In [3]:
from getpass import getpass
from datetime import datetime
from openai import OpenAI

api_key = getpass("DeepSeek API key（隐藏输入）：").strip()
if not api_key:
    raise ValueError("请输入密钥")
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com", timeout=60, max_retries=0)

trial = qd.complete_trial(trial, client)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
output_name = f"llm10_query_diagnosis_{run_id}.json"
mp.save_json([trial], output_name)
print("旧回答：", trial["old_answer"])
print("新回答：", trial["answer"])
print("状态：", trial["status"])


旧回答： 不对。手册未支持“电池模块放在仓库里就不用再充电”的说法。

手册摘录只提供了电池类型、模块数量、电压和 Ah 额定值等电池规格，以及更换电池需参阅适用电池更换用户手册/联系经销商或 APC 网站的信息；并未规定仓库储存时无需再充电。

资料未提供关于电池模块仓库储存是否需要充电的明确规定。
新回答： 不对。手册规定：贮存期间每六个月给电池模块充电一次。（EasyUPS_SRV_Chinese_990-91063B.pdf，PDF第4页，块02）
状态： ok


### 本次人工评价

核对新回答有没有纠正错误前提、说明具体周期并引用对应原文。以下保留本次判断，重跑后应根据新输出重新评价。

In [4]:
# 在这里添加trial["review"]，并写一句实际观察
trial["review"] = "通过：纠正了不用充电的错误前提，说明贮存期间每六个月充电一次，引用第4页块02包含对应依据。"

# 人工改写检索表达后，找到了原先遗漏的条款，新回答完整回答了原问题。

## 保存诊断与评价

评价写完后保存。这次人工改写得到正确答案，第9节首次检查仍保留4/5。这是进入调试后的单例诊断，不能代替新问题检查，也不能证明自动改写有效。

In [5]:
print("评价：", trial.get("review", "尚未填写"))
result_path = mp.save_json([trial], output_name)
print("保存路径：", result_path)


评价： 通过：纠正了不用充电的错误前提，说明贮存期间每六个月充电一次，引用第4页块02包含对应依据。
保存路径： D:\1rance\LLM\LLM_07_公开手册问答项目\results\llm10_query_diagnosis_20261013_131350_959665.json


## 复核记录

完整记录在 `results/llm10_query_diagnosis_20261013_131350_959665.json`，其中保留原问题、检索表达、两组候选、旧回答、新回答和人工评价。

原问题不变，人工调整检索表达，找到了原先遗漏的依据。下一轮需要其他新问题继续检查。